# InventoryPulse daily E&O
Synthetic prototype. See fabric/README.md for deployment prerequisites.

In [ ]:
from pathlib import Path
from datetime import date
import json
from inventorypulse.engine import run_pack

# Parameters: attach a default Lakehouse and the wheel's Fabric Environment first.
AS_OF = "2026-10-31"
INPUT = Path("/lakehouse/default/Files/inventorypulse/input")
run = run_pack(INPUT, date.fromisoformat(AS_OF))

# Explicit string schema handles empty arrays and stable currency serialization.
# For semantic modelling, cast money to decimal(20,2) and quantities to decimal(20,4).
from pyspark.sql.types import StructType, StructField, StringType

def persist(name, rows):
    if not rows:
        return
    cooked = [{"run_id": run["run_id"], "as_of_date": AS_OF, **row} for row in rows]
    fields = sorted({key for row in cooked for key in row})
    serial = [{k: (json.dumps(row.get(k)) if isinstance(row.get(k), (dict, list))
                   else str(row[k]) if row.get(k) is not None else None) for k in fields} for row in cooked]
    schema = StructType([StructField(k, StringType(), True) for k in fields])
    df = spark.createDataFrame(serial, schema=schema)
    # One reproducible partition per run. Reruns do not duplicate fact rows.
    (df.write.format("delta").mode("overwrite").partitionBy("run_id")
       .option("replaceWhere", f"run_id = '{run['run_id']}'").saveAsTable(name))

import csv
persist("source_holdings", list(csv.DictReader((INPUT / "holdings.csv").open())))
persist("source_sku_master", json.loads((INPUT / "sku_master.json").read_text()))
persist("source_labels", json.loads((INPUT / "labels.json").read_text()))
baseline = json.loads((INPUT / "legacy_baseline.json").read_text())
persist("source_legacy_baseline", [dict(baseline_cutoff=baseline["as_of_date"], **r) for r in baseline["positions"]])
for name in ("positions", "recommendations", "exceptions", "crosswalk", "alerts"):
    persist(name, run[name])
persist("reconciliation", run["reconciliation"]["rows"])
persist("run_summary", [dict(**run["summary"], alert_threshold=run["policy"]["alert_value"], data_mode=run["data_mode"])])
persist("scorecard", [run["scorecard"]])
persist("agent_handoffs", run["handoffs"])
persist("weekly_reports", [dict(role=k, report=v) for k, v in run["weekly_reports"].items()])
EVIDENCE = Path("/lakehouse/default/Files/inventorypulse/evidence") / run["run_id"]
EVIDENCE.mkdir(parents=True, exist_ok=True)
(EVIDENCE / "run.json").write_text(json.dumps(run, indent=2))
print(json.dumps(run["summary"], indent=2))
